# 11. Machine learning III — klasifikasi citra warna dan white balance

**Capaian:** Membandingkan model dengan/tanpa patch referensi abu-abu pada batch iluminasi berbeda.

**Prasyarat:** Python dasar, NumPy, dan membaca array. **Estimasi:** 75–100 menit.

> Kerjakan dari atas ke bawah. Setiap grafik wajib disertai penafsiran, bukan hanya ditempel.

## Konsep dan model

Simulasi kamera RGB memotret objek berwarna dan satu patch referensi netral. Setiap batch memiliki gain RGB berbeda. Gunakan fitur rerata kanal dan chromaticity: $r=R/(R+G+B)$. Faktor koreksi diagonal $R_{corr}=R\cdot 128/R_{ref}$ (serupa untuk G dan B). Ini bukan karakterisasi kamera lengkap maupun metode untuk mengukur $\Delta E_{00}$ terkalibrasi.

## Lihat iluminasi yang berubah

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pengolahan_citra_id.datasets import kolorimetri_sintetis
from pengolahan_citra_id.features import fitur_kolorimetri
from pengolahan_citra_id.evaluation import split_berdasar_batch,metrik_klasifikasi
images,y,batch=kolorimetri_sintetis(n_batch=24,per_batch=18,seed=42)
fig,ax=plt.subplots(2,4,figsize=(10,4))
for a,i in zip(ax.ravel(),[0,1,18,19,36,37,54,55]):
    a.imshow(images[i]);a.set_title(f'Kelas {y[i]}, batch {batch[i]}');a.axis('off')
plt.tight_layout();plt.show()

## Ekstraksi fitur tanpa dan dengan koreksi

In [ ]:
X_raw=np.array([fitur_kolorimetri(im,False) for im in images])
X_cor=np.array([fitur_kolorimetri(im,True) for im in images])
print('Fitur citra:',X_raw.shape)
print('Rerata RGB ternormalisasi raw:',X_raw[:4,:3].round(3))
print('Rerata RGB terkoreksi:',X_cor[:4,:3].round(3))

## Evaluasi held-out batches

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay
tr,te=split_berdasar_batch(y,batch,seed=42)
def fit_dan_uji(X):
    model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1500,random_state=42))
    model.fit(X[tr],y[tr]);p=model.predict(X[te])
    return model,p,metrik_klasifikasi(y[te],p)
model_raw,p_raw,m_raw=fit_dan_uji(X_raw)
model_cor,p_cor,m_cor=fit_dan_uji(X_cor)
print('Tanpa koreksi:',m_raw)
print('Dengan koreksi:',m_cor)
fig,ax=plt.subplots(1,2,figsize=(8,3))
ConfusionMatrixDisplay.from_predictions(y[te],p_raw,ax=ax[0]);ax[0].set_title('Raw')
ConfusionMatrixDisplay.from_predictions(y[te],p_cor,ax=ax[1]);ax[1].set_title('Koreksi')
plt.tight_layout();plt.show()

## Tinjau dampak per batch

In [ ]:
for b in sorted(set(batch[te])):
    ind=np.flatnonzero(batch[te]==b)
    print('Batch',int(b),'akurasi raw',round(np.mean(p_raw[ind]==y[te][ind]),3),
          '| terkoreksi',round(np.mean(p_cor[ind]==y[te][ind]),3))

## Diskusi, HOTS, dan latihan

1. Apakah white balance selalu meningkatkan F1 pada simulasi? Laporkan hasil sebenarnya.
2. Bedakan white balance, color correction matrix, dan transformasi CIELAB.
3. Susun desain eksperimen nyata: geometri lampu, paparan, ColorChecker, kartu abu-abu, dan batch lampu yang *unseen*.
4. Mengapa *random image split* berpotensi optimistis jika banyak citra satu sesi akuisisi?

**Laporan singkat:** rumusan masalah → rancangan eksperimen → hasil → interpretasi → keterbatasan → kesimpulan.

## Catatan keselamatan dan validitas

Objek dan label disimulasikan. Koreksi referensi abu-abu hanya ilustrasi normalisasi kanal, bukan prosedur kalibrasi warna ilmiah yang memadai.